# 🟡 Interview: Implement Activation Checkpointing

Trade compute for memory by recomputing activations during backward

In [ ]:
import torch


In [ ]:
# ✅ INTERVIEW

def checkpoint_sequential(fns, x):
    """
    手写梯度检查点 —— 面试版。

    面试考点:
    1. 激活检查点 = 用计算换内存
    2. 前向时只保存输入，不保存中间激活
    3. 反向时重新计算中间激活（额外一次前向）
    4. torch.utils.checkpoint.checkpoint 是标准实现

    参数: fns: list of callables, x: Tensor
    返回: Tensor
    """
    import torch.utils.checkpoint as cp

    # 逐层应用 checkpoint
    # 每一层的输入会被保存，但该层的中间激活不会
    # 反向传播到这一层时，会用保存的输入重新运行该层
    for fn in fns:
        x = cp.checkpoint(fn, x, use_reentrant=False)

    return x

In [ ]:
torch.manual_seed(0)
layers = [torch.nn.Linear(16, 16) for _ in range(4)]
x = torch.randn(4, 16, requires_grad=True)

# Checkpointed forward
out_cp = checkpoint_sequential(layers, x)

# Naive sequential forward
x2 = x.detach().requires_grad_(True)
out_naive = x2
for layer in layers:
    out_naive = layer(out_naive)

print("Outputs match:", torch.allclose(out_cp, out_naive, atol=1e-5))

# Confirm gradients flow through checkpointed path
out_cp.sum().backward()
print("Gradient flows (x.grad is not None):", x.grad is not None)

In [ ]:
from torch_judge import check
check("activation_checkpointing")

## 📝 核心思路总结

1. **以计算换内存**：前向传播时不保存中间激活值，反向传播时按需重新计算，内存从 O(N) 降到 O(√N)。
2. **torch.utils.checkpoint**：PyTorch 内置的检查点工具，`use_reentrant=False` 是推荐模式，支持更复杂的计算图。
3. **梯度必须能流回**：checkpoint 不影响梯度计算的正确性，只是延迟了中间激活的计算时机。
4. **适用场景**：深层网络（如 Transformer）训练时内存不足，但 GPU 算力有余量时使用。